In [1]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F
import numpy as np

# Configurando a semente para reprodutibilidade (Padrão de Pesquisa)
torch.manual_seed(42)

In [2]:
class ContinuousTokenizer(nn.Module):
    """
    Projeta variáveis contínuas escalares em um espaço vetorial de dimensão d.
    """
    def __init__(self, num_continuous_features: int, embed_dim: int):
        super().__init__()
        # Pesos: [num_features, embed_dim]
        self.weight = nn.Parameter(torch.randn(num_continuous_features, embed_dim))
        # Viés: [num_features, embed_dim]
        self.bias = nn.Parameter(torch.zeros(num_continuous_features, embed_dim))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x tem shape: [batch_size, num_continuous_features]
        # Adicionamos uma dimensão extra para fazer o broadcast da multiplicação
        # Resultado shape: [batch_size, num_continuous_features, embed_dim]
        return x.unsqueeze(2) * self.weight + self.bias

class CategoricalTokenizer(nn.Module):
    """
    Cria embeddings para variáveis categóricas, considerando diferentes cardinalidades.
    """
    def __init__(self, cardinalities: list, embed_dim: int):
        super().__init__()
        # ModuleList de Embeddings, um para cada coluna categórica
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=c, embedding_dim=embed_dim) 
            for c in cardinalities
        ])
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x tem shape: [batch_size, num_categorical_features]
        # Iteramos sobre cada coluna e seu respectivo embedding
        embedded_features = [
            embedding_layer(x[:, i]) 
            for i, embedding_layer in enumerate(self.embeddings)
        ]
        # Empilhamos os tensores. Shape final: [batch_size, num_features, embed_dim]
        return torch.stack(embedded_features, dim=1)

In [3]:
# ==========================================
# TESTANDO A MATEMÁTICA COM TENSORES SINTÉTICOS
# ==========================================
batch_size = 4
embed_dim = 16

# 1. Simulando 3 variáveis contínuas (ex: valor da transação, tempo de sessão, idade)
x_cont = torch.randn(batch_size, 3) 
tokenizer_cont = ContinuousTokenizer(num_continuous_features=3, embed_dim=embed_dim)
out_cont = tokenizer_cont(x_cont)

In [4]:
# 2. Simulando 2 variáveis categóricas (ex: Tipo de Dispositivo (3 tipos), Região (50 tipos))
x_cat = torch.tensor([[0, 45], [1, 12], [2, 3], [0, 49]]) # IDs aleatórios
tokenizer_cat = CategoricalTokenizer(cardinalities=[3, 50], embed_dim=embed_dim)
out_cat = tokenizer_cat(x_cat)

print(f"Shape de entrada contínua: {x_cont.shape}")
print(f"Shape após tokenização contínua: {out_cont.shape}\n")

print(f"Shape de entrada categórica: {x_cat.shape}")
print(f"Shape após tokenização categórica: {out_cat.shape}")

Shape de entrada contínua: torch.Size([4, 3])
Shape após tokenização contínua: torch.Size([4, 3, 16])

Shape de entrada categórica: torch.Size([4, 2])
Shape após tokenização categórica: torch.Size([4, 2, 16])


In [5]:
class TabularSelfAttention(nn.Module):
    """
    Mecanismo de Auto-Atenção Multi-Head projetado para dados tabulares.
    """
    def __init__(self, embed_dim: int, num_heads: int = 4):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        
        # A dimensão de cada "cabeça" de atenção
        assert embed_dim % num_heads == 0, "embed_dim deve ser divisível por num_heads"
        self.head_dim = embed_dim // num_heads
        
        # Matrizes de peso W_Q, W_K, W_V combinadas para otimização de memória
        # Mapeia de [embed_dim] para [3 * embed_dim]
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        
        # Projeção final de saída
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_features, _ = x.shape
        
        # 1. Projeção Linear: X * W
        # Shape: [Batch, Features, 3 * embed_dim]
        qkv = self.qkv_proj(x)
        
        # 2. Separação em Q, K, V e reestruturação para Multi-Head
        # Redimensiona para: [Batch, Features, 3, Heads, Head_Dim]
        qkv = qkv.reshape(batch_size, num_features, 3, self.num_heads, self.head_dim)
        
        # Permuta para: [3, Batch, Heads, Features, Head_Dim]
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # 3. Cálculo dos Scores de Atenção (Q * K^T / sqrt(d))
        # Multiplicação de matrizes em lote (Batched Matrix Multiplication)
        # q: [B, H, F, Head_Dim] @ k.transpose: [B, H, Head_Dim, F] -> [B, H, F, F]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # 4. Aplicação do Softmax para converter scores em probabilidades (Pesos)
        attention_weights = F.softmax(scores, dim=-1)
        
        # 5. Multiplicação pelos Valores (Weights * V)
        # [B, H, F, F] @ [B, H, F, Head_Dim] -> [B, H, F, Head_Dim]
        context = torch.matmul(attention_weights, v)
        
        # 6. Recombinação das cabeças (Concatenando de volta para embed_dim)
        # [B, F, H, Head_Dim] -> [B, F, embed_dim]
        context = context.permute(0, 2, 1, 3).reshape(batch_size, num_features, self.embed_dim)
        
        return self.out_proj(context), attention_weights

In [6]:
# ==========================================
# TESTANDO A INTEGRAÇÃO END-TO-END
# ==========================================
# Usando as saídas das camadas do passo anterior:
# out_cont (Shape: [4, 3, 16]) | out_cat (Shape: [4, 2, 16])

# Concatenamos as features contínuas e categóricas no mesmo tensor
# Shape final de entrada: [4, 5 features totais, 16]
x_combined = torch.cat([out_cont, out_cat], dim=1)

# Inicializando e rodando a Atenção
attention_layer = TabularSelfAttention(embed_dim=16, num_heads=4)
out_context, attn_weights = attention_layer(x_combined)

print(f"Shape do tensor combinado (Entrada): {x_combined.shape}")
print(f"Shape após Auto-Atenção (Saída): {out_context.shape}")
print(f"Shape da Matriz de Atenção: {attn_weights.shape} (Batch, Heads, Features, Features)")

Shape do tensor combinado (Entrada): torch.Size([4, 5, 16])
Shape após Auto-Atenção (Saída): torch.Size([4, 5, 16])
Shape da Matriz de Atenção: torch.Size([4, 4, 5, 5]) (Batch, Heads, Features, Features)


In [7]:
class Sparsemax(nn.Module):
    """
    Projeção euclidiana no simplex de probabilidade.
    Substitui o Softmax para gerar distribuições de peso estritamente esparsas (com zeros exatos).
    """
    def __init__(self, dim: int = -1):
        super().__init__()
        self.dim = dim

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # Movemos a dimensão alvo para o final para facilitar as operações de álgebra linear
        original_size = z.size()
        z = z.transpose(self.dim, -1)
        d = z.size(-1)

        # 1. Ordena os logits de forma decrescente
        z_sorted, _ = torch.sort(z, dim=-1, descending=True)
        
        # 2. Cria o vetor de índices (1, 2, ..., d)
        k_range = torch.arange(1, d + 1, device=z.device, dtype=z.dtype)
        
        # 3. Calcula a soma cumulativa para encontrar o limiar de corte (tau)
        z_cumsum = torch.cumsum(z_sorted, dim=-1)
        
        # 4. Condição matemática para encontrar o número de elementos não-nulos k(z)
        # 1 + k * z_{(k)} > sum_{j=1}^k z_{(j)}
        is_gt = (1.0 + k_range * z_sorted) > z_cumsum
        
        # Encontra o maior índice (k) onde a condição é verdadeira
        k = (is_gt * k_range).max(dim=-1, keepdim=True)[0]
        
        # 5. Calcula o limiar tau
        tau = (torch.gather(z_cumsum, -1, (k - 1).long()) - 1.0) / k
        
        # 6. Aplica a função ReLU (max(0, z - tau)) para zerar o resto
        p = torch.clamp(z - tau, min=0.0)
        
        return p.transpose(self.dim, -1)

class SparseTabularSelfAttention(nn.Module):
    """
    Auto-Atenção Multi-Head com mecanismo de seleção esparsa.
    """
    def __init__(self, embed_dim: int, num_heads: int = 4):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        # Substituímos o F.softmax por nossa nova camada matemática
        self.sparsemax = Sparsemax(dim=-1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_features, _ = x.shape
        qkv = self.qkv_proj(x).reshape(batch_size, num_features, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Aplicação da esparsidade geométrica
        attention_weights = self.sparsemax(scores)
        
        context = torch.matmul(attention_weights, v)
        context = context.permute(0, 2, 1, 3).reshape(batch_size, num_features, self.embed_dim)
        return self.out_proj(context), attention_weights

In [8]:
# ==========================================
# TESTANDO A ESPARSIDADE MATEMÁTICA
# ==========================================
# Usando o mesmo x_combined do passo anterior [4, 5, 16]
sparse_attention = SparseTabularSelfAttention(embed_dim=16, num_heads=4)
_, sparse_weights = sparse_attention(x_combined)

print("Matriz de Atenção da Cabeça 0 do primeiro registro (Softmax vs Sparsemax):")
print("\nRepare como o Sparsemax força os pesos irrelevantes para EXATAMENTE 0.0000:")
# Mostramos os pesos de interação da Feature 0 olhando para todas as outras 5 features
print(sparse_weights[0, 0, 0, :].detach().numpy())

Matriz de Atenção da Cabeça 0 do primeiro registro (Softmax vs Sparsemax):

Repare como o Sparsemax força os pesos irrelevantes para EXATAMENTE 0.0000:
[0.18141091 0.16378076 0.16939981 0.16327356 0.32213497]


In [9]:
class TabularFocalLoss(nn.Module):
    """
    Focal Loss focada em dados desbalanceados.
    Reduz o peso de gradientes para exemplos 'fáceis' e foca nos 'hard negatives'.
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # 1. Calculamos a Cross-Entropy clássica primeiro
        # Usamos F.cross_entropy porque ela embute a LogSoftmax, garantindo estabilidade numérica extrema.
        # reduction='none' nos devolve o erro individual de cada linha do lote.
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        
        # 2. Recuperamos a probabilidade pt exata da classe correta usando a inversa do log
        # Como ce_loss = -log(pt), então pt = exp(-ce_loss)
        pt = torch.exp(-ce_loss)
        
        # 3. Aplicamos a matemática da Focal Loss
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        # 4. Redução final
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [10]:
# ==========================================
# TESTE DE MESA: CROSS-ENTROPY VS FOCAL LOSS
# ==========================================
# Simulando um lote com 3 transações (Classes: 0 = Normal, 1 = Fraude)
# Transação 1: Normal (Fácil, modelo tem certeza)
# Transação 2: Normal (Fácil, modelo tem certeza)
# Transação 3: FRAUDE (Difícil, modelo achou que era normal)

# Logits puros (saída da última camada linear antes do Softmax)
logits = torch.tensor([
    [ 2.5, -1.0], # Modelo dá alta probabilidade para classe 0 (Normal)
    [ 3.0, -2.0], # Modelo dá alta probabilidade para classe 0 (Normal)
    [ 1.5, -0.5]  # Modelo dá alta probabilidade para classe 0, mas ERA FRAUDE!
])

# Alvos reais (Gabarito)
targets = torch.tensor([0, 0, 1]) 

# Calculando perdas
ce_criterion = nn.CrossEntropyLoss(reduction='none')
fl_criterion = TabularFocalLoss(alpha=1.0, gamma=2.0, reduction='none')

ce_erros = ce_criterion(logits, targets)
fl_erros = fl_criterion(logits, targets)

print("Comparação de Erros por Transação:\n")
for i in range(3):
    print(f"Transação {i+1} (Alvo={targets[i].item()}):")
    print(f"  Cross-Entropy Clássica: {ce_erros[i]:.4f}")
    print(f"  Focal Loss (Avançada) : {fl_erros[i]:.4f}")
    print("-" * 40)

Comparação de Erros por Transação:

Transação 1 (Alvo=0):
  Cross-Entropy Clássica: 0.0298
  Focal Loss (Avançada) : 0.0000
----------------------------------------
Transação 2 (Alvo=0):
  Cross-Entropy Clássica: 0.0067
  Focal Loss (Avançada) : 0.0000
----------------------------------------
Transação 3 (Alvo=1):
  Cross-Entropy Clássica: 2.1269
  Focal Loss (Avançada) : 1.6501
----------------------------------------


In [11]:
import numpy as np

def calibracao_conformal(probs_calibracao: np.ndarray, labels_calibracao: np.ndarray, alpha: float = 0.10):
    """
    Calcula o limiar de incerteza (q_hat) usando um conjunto de calibração separado.
    alpha=0.10 garante estatisticamente 90% de cobertura.
    """
    # 1. Calculamos o "Score de Não-Conformidade" (quão confiante o modelo estava na resposta errada)
    # Se label=1, o erro é (1 - prob). Se label=0, o erro é a própria prob.
    scores_erro = np.where(labels_calibracao == 1, 1 - probs_calibracao, probs_calibracao)
    
    # 2. Ordenamos os scores para encontrar o limite de contenção
    n = len(scores_erro)
    scores_ordenados = np.sort(scores_erro)
    
    # 3. Fórmula do Quantil Conformal: p = (n + 1) * (1 - alpha) / n
    indice_quantil = int(np.ceil((n + 1) * (1 - alpha)))
    indice_quantil = min(indice_quantil, n - 1)
    
    q_hat = scores_ordenados[indice_quantil]
    return q_hat

In [12]:
# ==========================================
# TESTE DE MESA: CONFORMAL PREDICTION
# ==========================================
# Simulando previsões do modelo em dados que ele nunca viu (Conjunto de Calibração)
probs_modelo = np.array([0.1, 0.2, 0.85, 0.9, 0.4, 0.95, 0.05, 0.3])
gabarito_real = np.array([0,   0,   1,    1,   0,   1,    0,    1]) # O último 1 (fraude) o modelo errou feio (0.3)

# Calculando a margem para 90% de confiança
margem_q_hat = calibracao_conformal(probs_modelo, gabarito_real, alpha=0.10)

print(f"Limiar de Correção Conformal (q_hat): {margem_q_hat:.4f}")
print("\nRegra de Inferência para Produção:")
print(f"Se a previsão do modelo oscilar em +/- {margem_q_hat:.4f}, o sistema deve acionar intervenção humana pois não há garantia estatística.")

Limiar de Correção Conformal (q_hat): 0.7000

Regra de Inferência para Produção:
Se a previsão do modelo oscilar em +/- 0.7000, o sistema deve acionar intervenção humana pois não há garantia estatística.
